In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class LSTM(torch.nn.Module):
    def __init__(self, input_dim, emb_dim,hidden_dim ):
        super().__init__()
        self.hid_dim = hidden_dim
        self.embedding_dim = emb_dim
        self.input_dim = input_dim

        # self.longterm = torch.tensor(torch.randn(self.hid_dim),requires_grad=False) 
        # self.hidden_state_init = False
        # self.short_term = torch.tensor(torch.randn(self.hid_dim),requires_grad=False)
        self.uf = nn.Linear(self.input_dim,self.hid_dim,bias=False) # forget gate new token input
        self.wf = nn.Linear(self.hid_dim,self.hid_dim,bias=False) # forget gate hidden state input
        self.bias_forget = nn.Parameter(torch.zeros(self.hid_dim))
        self.input_u = nn.Linear(self.input_dim,self.hid_dim,bias=False) # input gate new tok
        self.input_w = nn.Linear(self.hid_dim,self.hid_dim,bias=False) # input gate state
        self.bias_input = nn.Parameter(torch.zeros(self.hid_dim)) #input gate bias
        self.input_tan_u = nn.Linear(self.input_dim,self.hid_dim,bias=False)
        self.input_tan_w = nn.Linear(self.hid_dim,self.hid_dim,bias=False)
        self.input_tan_bias = nn.Parameter(torch.zeros(self.hid_dim))
        # self.output = nn.Linear(self.input_dim+self,self.hid_dim,self.hid_dim)
        self.pot_short_term_mem = nn.Linear(self.hid_dim,self.hid_dim,bias=False)
        self.perc_short_term_mem_u = nn.Linear(self.input_dim,self.hid_dim,bias=False)
        self.perc_short_term_mem_w = nn.Linear(self.hid_dim,self.hid_dim,bias=False)
        self.perc_short_term_mem_bias = nn.Parameter(torch.zeros(self.hid_dim))

    # def init_state(self,batch_size,device):
    #     if self.hidden_state_init ==False:
    #         first_hidden_state = torch.zeros(batch_size,self.hid_dim,requires_grad=False )
    #         self.hidden_state_init = True
    #         return first_hidden_state
    def forward(self,x,batch,seqlen,device,state_tuple=None,):
        if state_tuple is None:
            # Initialize fresh zeros if no state is provided
            h_t = torch.zeros(batch, self.hid_dim)
            c_t = torch.zeros(batch, self.hid_dim)
        else:
            h_t, c_t = state_tuple
        outputs=[]
        longterm_history=[]
        for i in range(seqlen):
   
            current_x = x[:, i, :]
            bef_forget = (self.uf(current_x) + self.wf(h_t)) + self.bias_forget
            forget = torch.sigmoid(bef_forget)




            #  = torch.cat(,state, dim = 0)
            #  = self.wi1()
            #  = torch.sigmoid()
            c_t = c_t*forget
            input_g = torch.sigmoid((self.input_u(current_x) + self.input_w(h_t)) + self.bias_input)
            propose = torch.tanh(((self.input_tan_u(current_x) + self.input_tan_w(h_t)) + self.input_tan_bias))
  
            c_t += input_g * propose
            short_term_perc = torch.sigmoid((self.perc_short_term_mem_u(current_x) + self.perc_short_term_mem_w(h_t)) + self.perc_short_term_mem_bias)
            pot_short = torch.tanh(c_t)
            new_short_term = short_term_perc * pot_short
            h_t = new_short_term
            outputs.append(h_t)
            longterm_history.append(c_t)
            # full_short_terms = torch.tensor(batch,seqlen,self.hid_dim)
            # full_short_terms = torch.cat((full_short_terms,h_t,full_short_terms), dim= 1)
            # full_long_terms =torch.tensor(batch,seqlen,self.hid_dim) 
            # torch.cat((full_long_terms,c_t,full_long_terms),dim=1)
        final_long_term_sequence = torch.stack(longterm_history,dim=1)
        final_output_sequence = torch.stack(outputs,dim=1)
        return final_output_sequence[:,-1,:],final_long_term_sequence[:,-1,:],final_output_sequence,final_long_term_sequence




            



            # # i fucked up the order of things the hidden state should 
            # be gotten from the residual stream and multiplied by weights and then
            #  the inputs after being multiplied by their own weights should be added to the cooked state

        







    
